# Spotify Top 50 Mexico Analysis

In this notebook I'll analyze a sample of 15 tracks from Spotify's Top 50 Mexico chart.  
This dataset is interesting because it involves multiple arrays: streams, BPM, duration 

**Source:** Spotify Charts — https://charts.spotify.com/charts/view/regional-mx-weekly/latest  
**BPM data:** https://tunebat.com

---

In [2]:
import numpy as np

In [3]:
# --- Data ---
tracks = np.array([
    'La Bebe', 'Ella Baila Sola', 'Shakira: Bzrp Music Sessions',
    'Quevedo: Bzrp Music Sessions', 'TQM', 'Un Verano Sin Ti',
    'La Fama', 'Hawai', 'Tití Me Preguntó', 'Me Porto Bonito',
    'Provenza', 'Ojitos Lindos', 'Efecto', 'Calle de tu casa', 'Kaza'
])

# Streams in millions
streams = np.array([312.4, 487.2, 893.1, 654.3, 198.7, 421.6, 267.9,
                    534.8, 445.2, 398.1, 189.4, 356.7, 223.5, 144.8, 178.3])

# Beats per minute
bpm = np.array([95, 105, 128, 115, 88, 97, 118, 92, 110, 102,
                85, 99, 112, 94, 108])

# Duration in seconds
duration_sec = np.array([187, 214, 198, 231, 175, 203, 245, 192, 218, 207,
                          238, 196, 183, 221, 169])

---
## Part 1 — Stream Statistics

Let's get an insight of how streams are distributed across the sample.

In [13]:
# Let's find the most and least streamed track (name + value),
print(f"Most streamed track: {tracks[streams.argmax()]}")
print(f"Least streamed track: {tracks[streams.argmin()]}")

# the average, median and standard deviation of streams
print(f"Average number of streams: {streams.mean():.2f}")
print(f"Median number of streams: {np.median(streams):.2f}")
print(f"Standart deviation number of streams: {streams.std():.2f}")

# Also: is the mean higher than the median? What does that tell us about the distribution?
print(f"Mean higher than median? {streams.mean() > np.median(streams)}")
print(f"Right skewed distribution, a few hits had a lot of streams")


Most streamed track: Shakira: Bzrp Music Sessions
Least streamed track: Calle de tu casa
Average number of streams: 380.40
Median number of streams: 356.70
Standart deviation number of streams: 197.93
Mean higher than median? True
Right skewed distribution, a few hits had a lot of streams


In [ ]:
# Let's calculate what percentage of total streams each track represent
# stream_share = (track_streams / total_streams) * 100
stream_share = (streams / streams.sum()) * 100
print(f"Streams share: {np.round(stream_share,2)}")

# Print the top 3 tracks with their share percentage
sorted_streams = np.argsort(stream_share)[::-1]
s=0
while s < 3:
    print(f"Track number {s+1}: {tracks[sorted_streams[s]]} -> {np.round(stream_share[sorted_streams[s]],2)} streams")
    s = s+1


Streams share: [ 5.47  8.54 15.65 11.47  3.48  7.39  4.7   9.37  7.8   6.98  3.32  6.25
  3.92  2.54  3.12]
Track number 1: Shakira: Bzrp Music Sessions -> 15.65 streams
Track number 2: Quevedo: Bzrp Music Sessions -> 11.47 streams
Track number 3: Hawai -> 9.37 streams


---
## Part 2 — Ranking with argsort

One of the most practical NumPy tools for real data: sorting and ranking.

In [28]:
# Print the full ranking from most to least streamed using argsort
# Format: Rank | Track name | Streams
s=0
while s < len(sorted_streams):
    print(f"Track number {s+1}: {tracks[sorted_streams[s]]} -> {np.round(stream_share[sorted_streams[s]],2)} streams")
    s = s+1


Track number 1: Shakira: Bzrp Music Sessions -> 15.65 streams
Track number 2: Quevedo: Bzrp Music Sessions -> 11.47 streams
Track number 3: Hawai -> 9.37 streams
Track number 4: Ella Baila Sola -> 8.54 streams
Track number 5: Tití Me Preguntó -> 7.8 streams
Track number 6: Un Verano Sin Ti -> 7.39 streams
Track number 7: Me Porto Bonito -> 6.98 streams
Track number 8: Ojitos Lindos -> 6.25 streams
Track number 9: La Bebe -> 5.47 streams
Track number 10: La Fama -> 4.7 streams
Track number 11: Efecto -> 3.92 streams
Track number 12: TQM -> 3.48 streams
Track number 13: Provenza -> 3.32 streams
Track number 14: Kaza -> 3.12 streams
Track number 15: Calle de tu casa -> 2.54 streams


In [35]:
# Now rank by BPM (fastest to slowest)
# Does the fastest song also have the most streams?
sorted_bpm = np.argsort(bpm)[::-1]
s=0
while s < len(sorted_bpm):
    print(f"{tracks[sorted_bpm[s]]} -> BPM {np.round(bpm[sorted_bpm[s]],2)}")
    s = s+1

print("Fastest track:", tracks[sorted_bpm[0]], f"({bpm[sorted_bpm[0]]} BPM)")
print("Does it have most streams?", tracks[sorted_bpm[0]] == tracks[streams.argmax()])

Shakira: Bzrp Music Sessions -> BPM 128
La Fama -> BPM 118
Quevedo: Bzrp Music Sessions -> BPM 115
Efecto -> BPM 112
Tití Me Preguntó -> BPM 110
Kaza -> BPM 108
Ella Baila Sola -> BPM 105
Me Porto Bonito -> BPM 102
Ojitos Lindos -> BPM 99
Un Verano Sin Ti -> BPM 97
La Bebe -> BPM 95
Calle de tu casa -> BPM 94
Hawai -> BPM 92
TQM -> BPM 88
Provenza -> BPM 85
Fastest track: Shakira: Bzrp Music Sessions (128 BPM)
Does it have most streams? True


---
## Part 3 — BPM & Energy Classification

BPM is a proxy for energy level. Let's group the tracks and see if energy correlates with streams.

In [42]:
# Classify tracks by energy:
# High energy  → BPM >= 110
# Mid energy   → 95 <= BPM < 110
# Low energy   → BPM < 95
high_energy = bpm >= 110
mid_energy = (bpm >= 95) & (bpm <110)
low_energy = bpm < 95

# Print each group with track names
print(f"High energy tracks: {tracks[high_energy]}")
print(f"Mid energy tracks: {mid_energy[mid_energy]}")
print(f"Low energy tracks: {low_energy[low_energy]}")





High energy tracks: ['Shakira: Bzrp Music Sessions' 'Quevedo: Bzrp Music Sessions' 'La Fama'
 'Tití Me Preguntó' 'Efecto']
Mid energy tracks: [ True  True  True  True  True  True]
Low energy tracks: [ True  True  True  True]


In [47]:
# For each energy group, calculate the average streams
print(f"Avg stream High: {streams[high_energy].mean():.2f}")
print(f"Avg stream Mid: {streams[mid_energy].mean():.2f}")
print(f"Avg stream Low: {streams[low_energy].mean():.2f}")

# Which energy level tends to perform better on this chart?
print(f"Best lever energy by performance: High Energy")

Avg stream High: 496.80
Avg stream Mid: 359.05
Avg stream Low: 266.93
Best lever energy by performance: High Energy


---
## Part 4 — Duration Analysis

Streaming platforms have changed song lengths. Let's see what the data says.

In [54]:
# Convert duration_sec to minutes
# Print the longest and shortest track with their duration in minutes
# Print the average duration
duration_min = duration_sec / 60
print(f"Longest track: {tracks[duration_min.argmax()]} -> {duration_min.max():.2f}")
print(f"Shortest track: {tracks[duration_min.argmin()]} -> {duration_min.min():.2f}")
print(f"Average duration: {duration_min.mean():.2f}")

Longest track: La Fama -> 4.08
Shortest track: Kaza -> 2.82
Average duration: 3.42


In [67]:
# Calculate a 'streams per second' metric:
# streams_per_sec = (streams * 1_000_000) / duration_sec
# This rewards songs that accumulate many streams relative to their length
streams_per_sec = (streams * 1000000) / duration_sec

# Rank the top 5 by this metric — is it the same ranking as raw streams?
rank_streams_second = np.argsort(streams_per_sec[::-1])
s = 0
while s < 5:
    print(f"Track{[s+1]}: {tracks[rank_streams_second[s]]} -> {np.round(streams_per_sec[rank_streams_second[s]],2)}")
    s = s+1


Track[1]: Ella Baila Sola -> 2276635.51
Track[2]: TQM -> 1135428.57
Track[3]: La Bebe -> 1670588.24
Track[4]: Tití Me Preguntó -> 2042201.83
Track[5]: Provenza -> 795798.32


---
## Part 5 — Multi-condition Filters

In [68]:
# Filter 1: tracks with above-average streams AND BPM >= 100
# These are the high-energy tracks that are also commercially successful
filter1 = (streams > streams.mean()) & (bpm >=100)
print(f"Above avg streams and BPM >= 100: {tracks[filter1]}")


Above avg streams and BPM >= 100: ['Ella Baila Sola' 'Shakira: Bzrp Music Sessions'
 'Quevedo: Bzrp Music Sessions' 'Tití Me Preguntó' 'Me Porto Bonito']


In [69]:
# Filter 2: tracks that are short (< 200 sec) AND have high streams (> 300M)
# Short AND popular — the streaming-optimized formula
filter2 = (streams > 300) & (duration_sec < 200)
print(f"High streams and duration < 200: {tracks[filter2]}")

High streams and duration < 200: ['La Bebe' 'Shakira: Bzrp Music Sessions' 'Hawai' 'Ojitos Lindos']


In [70]:
# Filter 3 — let's invert the logic:
# Which tracks have BELOW-average streams despite having high BPM (>= 105)?
# High energy but underperforming commercially
filter3 = (streams < streams.mean()) & (bpm >=105)
print(f"Below avg streams and BPM >= 105: {tracks[filter3]}")

Below avg streams and BPM >= 105: ['La Fama' 'Efecto' 'Kaza']


---
## Part 6 — Going Further

In [80]:
# Use np.percentile() to find the 25th, 50th and 75th percentile of streams
# Then classify each track as: 'top quartile', 'upper mid', 'lower mid', 'bottom quartile'
# based on which percentile range it falls in
p25, p50, p75 = np.percentile(streams, [25, 50, 75])
for n,s in zip(tracks,streams):
    if s > p75:
        print(f"{n}: Top Quartile")
    elif s > p50:
        print(f"{n}: Upper Mid Quartile")
    elif s > p25:
        print(f"{n}: Lower Mid Quartile")
    else:
        print(f"{n}: Bottom Quartile")


La Bebe: Lower Mid Quartile
Ella Baila Sola: Top Quartile
Shakira: Bzrp Music Sessions: Top Quartile
Quevedo: Bzrp Music Sessions: Top Quartile
TQM: Bottom Quartile
Un Verano Sin Ti: Upper Mid Quartile
La Fama: Lower Mid Quartile
Hawai: Top Quartile
Tití Me Preguntó: Upper Mid Quartile
Me Porto Bonito: Upper Mid Quartile
Provenza: Bottom Quartile
Ojitos Lindos: Lower Mid Quartile
Efecto: Lower Mid Quartile
Calle de tu casa: Bottom Quartile
Kaza: Bottom Quartile


In [86]:
# Finally, normalize streams and BPM to 0-1 scale (Min-Max normalization)
# Then calculate a simple 'score' = 0.7 * norm_streams + 0.3 * norm_bpm
# This weights commercial success more than energy
# Which track has the highest combined score?

norm_streams = (streams - streams.min())/(streams.max() - streams.min())
norm_bpm = (bpm - bpm.min())/(bpm.max()-bpm.min())

score = 0.7*norm_streams + 0.3*norm_bpm
for t,s in zip(tracks,score):
    print(f"{t} : {s:.2f}")

print(f"Highest combined score: {tracks[score.argmax()]} -> {score.max():.2f}")

La Bebe : 0.23
Ella Baila Sola : 0.46
Shakira: Bzrp Music Sessions : 1.00
Quevedo: Bzrp Music Sessions : 0.69
TQM : 0.07
Un Verano Sin Ti : 0.34
La Fama : 0.35
Hawai : 0.41
Tití Me Preguntó : 0.46
Me Porto Bonito : 0.36
Provenza : 0.04
Ojitos Lindos : 0.30
Efecto : 0.26
Calle de tu casa : 0.06
Kaza : 0.19
Highest combined score: Shakira: Bzrp Music Sessions -> 1.00


---
## Observations

*Write your findings here after completing the analysis.*

- Shakira's BZRP session dominates the sample with nearly 900M streams, almost double the second place.
- The average track in this sample runs just over 3 minutes, consistent with streaming optimized song lengths.
- High-energy tracks (≥110 BPM) don't necessarily correlate with more streams 
- The streams-per-second metric reveals that shorter tracks can punch above their weight in engagement.